
# Python 核心数据结构：列表、元组、字典、集合

作为 Java 开发者，理解 Python 的核心数据结构最快的方式就是**“对比 Java 集合框架”**。Python 的这四大容器在底层设计上与 Java 有着惊人的相似之处，但 Python 在语法层面提供了更多的“语法糖”。

以下为你详细拆解这四种核心数据结构的底层原理与常用操作：

## 1. 列表 (list) —— 动态数组

- **Java 映射**：`ArrayList<E>`
- **特性**：有序、可变、允许重复元素。
- **底层原理**：
    Python 列表底层是一个**动态数组**（由 `PyListObject` 结构体实现）。它在内存中维护了一个连续的指针数组，每个指针指向实际的元素对象。当插入元素导致容量不足时，Python 会触发**扩容机制**，采用约 12.5% 冗余的几何增长策略（类似 Java 的 1.5 倍扩容），从而保证 `append()` 操作的均摊时间复杂度为 O(1)。需要注意的是，Python 列表**没有主动缩容机制**，删除元素只改变长度，不释放底层内存。
- **常用操作**：
    - 增：`append(x)` 末尾添加，`insert(i, x)` 指定位置插入，`extend(iterable)` 批量追加。
    - 删：`pop(i)` 弹出指定索引元素，`remove(x)` 删除首个匹配元素，`clear()` 清空。
    - 查/改：`list[i]` 索引访问（O(1)），`list[start:stop:step]` 切片操作。
    - 排：`sort()` 原地排序，`reverse()` 原地反转。

## 2. 元组 (tuple) —— 不可变数组

- **Java 映射**：无直接对应（类似 `Collections.unmodifiableList()` 或 Java 14+ 的 `Record`）。
- **特性**：有序、**不可变**、允许重复元素。
- **底层原理**：
    元组底层由 `PyTupleObject` 结构体实现，同样是指针数组，但**没有容量（allocated）的概念**，因为创建后大小固定，不支持 resize。由于不可变，元组在内存中占用更小，创建速度比列表快约 30%。此外，CPython 内部对长度 ≤20 的小元组使用了**缓存池（free_list）机制**，复用内存以减少分配开销。因为不可变，只要内部元素可哈希，元组自身就是**可哈希的**。
- **常用操作**：
    - 创建：`(1, 2, 3)`，注意单元素元组必须加逗号 `(1,)`。
    - 查：`t[i]` 索引访问，`t[1:3]` 切片。
    - 统计：`count(x)` 统计出现次数，`index(x)` 查找首次出现的索引。
    - 运算：`+` 拼接，`*` 重复。
- **生产场景**：函数返回多个值、作为字典的键（Key）、数据库查询结果集。

## 3. 字典 (dict) —— 哈希表

- **Java 映射**：`LinkedHashMap<K,V>` (Python 3.7+) / `HashMap<K,V>`
- **特性**：键值对（Key-Value）存储、**键必须唯一且不可变**、Python 3.7+ 保证插入顺序。
- **底层原理**：
    Python 字典底层基于**动态哈希表**（散列表），采用**开放寻址法**（而非 Java 的链地址法）解决哈希冲突。当发生冲突时，通过伪随机扰动探测序列寻找下一个空槽位。字典的负载因子达到约 2/3 时会触发 2 倍扩容并进行重哈希。这种“空间换时间”的设计使得字典的增删查改平均时间复杂度均为 O(1)。
- **常用操作**：
    - 增/改：`dict[key] = value`，`update(other_dict)` 批量更新。
    - 删：`pop(key, default)` 安全删除并返回值，`del dict[key]`。
    - 查：`dict.get(key, default)` 推荐的安全获取方式（不存在不报错）。
    - 遍历：`keys()`, `values()`, `items()` 获取视图对象。

## 4. 集合 (set) —— 去重哈希表

- **Java 映射**：`HashSet<E>`
- **特性**：**无序**、**元素唯一（自动去重）**、元素必须可哈希。
- **底层原理**：
    集合本质上是一个**只存储 Key、不存储 Value 的简化版字典**。它同样基于开放寻址法实现的哈希表，内部维护元素的指针和缓存的哈希值。当负载因子 ≥ 0.66 时自动扩容。由于底层也是哈希表，它的成员检测（`in` 操作）时间复杂度为 O(1)，远快于列表的 O(n)。
- **常用操作**：
    - 增/删：`add(x)`, `remove(x)` (不存在报错), `discard(x)` (安全删除)。
    - 集合运算：并集 `A | B`，交集 `A & B`，差集 `A - B`，对称差集 `A ^ B`。
    - 关系测试：`issubset()` 子集判断，`issuperset()` 超集判断。
- **生产场景**：列表数据去重、高频成员检测、权限/标签集合运算。

---

## 💡 核心选型指南（Java 开发者速记）

| 业务需求 | 选择类型 | 核心原因 |
| :--- | :--- | :--- |
| 存储多条同构记录，需按索引访问 | `list` | 对应 `ArrayList`，支持动态扩容和切片 |
| 存储配置、JSON 对象、键值映射 | `dict` | 对应 `HashMap`，O(1) 查找，键值对结构 |
| 函数返回多个值、固定配置常量 | `tuple` | 不可变、内存小、可哈希（能做 dict 的 key） |
| 数据去重、判断元素是否存在、集合运算 | `set` | 对应 `HashSet`，O(1) 成员检测，自动去重 |

**⚠️ 避坑提醒**：
在 Python 中，**只有不可变类型（如 int, str, tuple）才能作为字典的 Key 或放入 Set 中**。如果你把 `list` 或 `dict` 作为 Key，会直接抛出 `TypeError: unhashable type`。